# Bayesian Networks

**Companion lesson:** https://ml-viz.vercel.app/courses/graphical-models/01-bayesian-networks

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## The Sprinkler network — exact inference from scratch

Variables: Cloudy $C$, Sprinkler $S$, Rain $R$, Wet grass $W$, with $P(C,S,R,W)=P(C)P(S|C)P(R|C)P(W|S,R)$. We enumerate the full joint and answer queries by brute force — no library.

In [ ]:
P_C = [0.5, 0.5]                       # P(C=0), P(C=1)
P_S = [[0.5, 0.5], [0.9, 0.1]]          # P(S=s | C=c)
P_R = [[0.8, 0.2], [0.2, 0.8]]          # P(R=r | C=c)
# P(W=w | S=s, R=r)
P_W = [[[1.0, 0.0], [0.1, 0.9]], [[0.1, 0.9], [0.01, 0.99]]]

def joint(c, s, r, w):
    return P_C[c] * P_S[c][s] * P_R[c][r] * P_W[s][r][w]

# sanity: full joint sums to 1
total = sum(joint(c,s,r,w) for c in (0,1) for s in (0,1) for r in (0,1) for w in (0,1))
print('joint sums to', round(total, 6))

## Querying by enumeration

$P(R=1\mid W=1)=\dfrac{\sum_{c,s}P(c,s,R{=}1,W{=}1)}{\sum_{c,s,r}P(c,s,r,W{=}1)}$ — sum out the unobserved variables, then normalize.

In [ ]:
def query(evidence, target):
    """P(target_var = target_val | evidence dict)."""
    tv, tval = target
    num = den = 0.0
    for c in (0,1):
        for s in (0,1):
            for r in (0,1):
                for w in (0,1):
                    assign = dict(C=c, S=s, R=r, W=w)
                    if any(assign[k] != v for k, v in evidence.items()):
                        continue
                    p = joint(c, s, r, w)
                    den += p
                    if assign[tv] == tval: num += p
    return num / den

print('P(Rain=1)            =', round(query({}, ('R',1)), 3))
print('P(Rain=1 | Wet=1)    =', round(query({'W':1}, ('R',1)), 3), ' (wet grass raises rain)')

## Explaining away

Rain and Sprinkler are marginally independent causes of wet grass. Once we observe wet grass, learning the sprinkler was on should **lower** the probability of rain.

In [ ]:
p_rain_wet      = query({'W':1}, ('R',1))
p_rain_wet_spr  = query({'W':1, 'S':1}, ('R',1))
print(f'P(Rain | Wet)            = {p_rain_wet:.3f}')
print(f'P(Rain | Wet, Sprinkler) = {p_rain_wet_spr:.3f}')
print('-> sprinkler explains away the wet grass, so rain becomes less likely.')

## Parameter savings from factorization

In [ ]:
full = 2**4 - 1
factored = 1 + 2 + 2 + 4    # P(C) + P(S|C) + P(R|C) + P(W|S,R)
print(f'full joint table: {full} free parameters | factorized: {factored}')

## Key takeaways

- A Bayesian network factorizes the joint into per-node conditionals over a DAG.
- Exact inference by enumeration: sum out unobserved variables, then normalize.
- Observing a common effect couples its causes — **explaining away**.
- Factorization turns an exponential table into a few small CPTs.